Reference Paper: [The Link Prediction Problem for Social Networks](https://www.cs.cornell.edu/home/kleinber/link-pred.pdf)

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

dataset = "dataset/CollegeMsg.txt"

df = pd.read_csv(dataset, sep=' ', names = ["src", "tgt", "time"]).sort_values("time")

# Split dataset
split_ratio = 0.8
split_idx = int(split_ratio * len(df))
train_df = df.iloc[:split_idx]
test_df  = df.iloc[split_idx:]

In [2]:
# Build graph
G = nx.Graph()
G.add_edges_from(zip(train_df["src"], train_df["tgt"]))
nodes = list(G.nodes())

# Only using active nodes as per the paper
k = 3 #k_trainig and k_test
Core = set([n for n, d in G.degree() if d >= k])

# Candidate pairs to predict from the Core set
def get_candidate_pairs(G, Core):
    candidates = []
    for u in Core:
        for v in Core:
            if u < v and not G.has_edge(u, v):
                candidates.append((u, v))
    return candidates
candidates = get_candidate_pairs(G, Core)

print(len(Core))
print(len(candidates))

1158
658983


In [3]:
# Metrics as per the paper

sp = dict(nx.all_pairs_shortest_path_length(G, cutoff=6))
def graph_distance(u, v):
    return -sp.get(u, {}).get(v, np.inf)  # negative (closer = better)

def common_neighbors(u, v):
    return len(list(nx.common_neighbors(G, u, v)))

jaccard_scores = {(u,v):p for u,v,p in nx.jaccard_coefficient(G, candidates)}

adamic_scores = {(u,v):p for u,v,p in nx.adamic_adar_index(G, candidates)}

pa_scores = {(u,v):p for u,v,p in nx.preferential_attachment(G, candidates)}

# Katz beta
beta = 0.005
A = nx.to_numpy_array(G)
I = np.eye(A.shape[0])
K = np.linalg.inv(I - beta * A) - I  # Katz matrix
node_index = {n:i for i,n in enumerate(G.nodes())}
def katz(u, v):
    return K[node_index[u], node_index[v]]

# Rooted PageRank
alpha = 0.15
def rooted_pagerank(u):
    personalization = {n:0 for n in G.nodes()}
    personalization[u] = 1
    return nx.pagerank(G, alpha=alpha, personalization=personalization)
rpr_cache = {}
def rpr(u, v):
    if u not in rpr_cache:
        rpr_cache[u] = rooted_pagerank(u)
    return rpr_cache[u].get(v, 0)

# SimRank
def simrank(G, C=0.8, max_iter=25):
    nodes = list(G.nodes())
    sim = {u:{v:(1 if u==v else 0) for v in nodes} for u in nodes}
    for _ in range(max_iter):
        new_sim = {}
        for u in nodes:
            new_sim[u] = {}
            for v in nodes:
                if u == v:
                    new_sim[u][v] = 1
                else:
                    Nu = list(G.neighbors(u))
                    Nv = list(G.neighbors(v))
                    if len(Nu)==0 or len(Nv)==0:
                        new_sim[u][v] = 0
                    else:
                        s = sum(sim[x][y] for x in Nu for y in Nv)
                        new_sim[u][v] = (C * s) / (len(Nu)*len(Nv))
        sim = new_sim
    return sim
sim = simrank(G)

# Hitting & Compute time
P = nx.to_numpy_array(G)
P = P / P.sum(axis=1, keepdims=True)
n = P.shape[0]
Z = np.linalg.pinv(np.eye(n) - P + np.ones((n,n))/n) # Fundamental matrix approximation
pi = P.sum(axis=0) # Stationary distribution
pi = pi / pi.sum()
def hitting_time(i, j):
    return (Z[j,j] - Z[i,j]) / pi[j]
def commute_time(i, j):
    return hitting_time(i,j) + hitting_time(j,i)

In [4]:
def compute_scores():
    scores = {
        "Common Neighbours": [],
        "Jaccard’s coefficient": [],
        "Adamic": [],
        "Preferential Attachment": [],
        "Graph Distance": [],
        "Katz": [],
        "Rooted PageRank": [],
        "SimRank": [],
        "Hitting Time": [],
        "Commute Time": [],
    }

    for u,v in candidates:
        i, j = node_index[u], node_index[v]

        scores["Common Neighbours"].append((u,v, common_neighbors(u,v)))
        scores["Jaccard’s coefficient"].append((u,v, jaccard_scores.get((u,v),0)))
        scores["Adamic"].append((u,v, adamic_scores.get((u,v),0)))
        scores["Preferential Attachment"].append((u,v, pa_scores.get((u,v),0)))
        scores["Graph Distance"].append((u,v, graph_distance(u,v)))
        scores["Katz"].append((u,v, katz(u,v)))
        scores["Rooted PageRank"].append((u,v, rpr(u,v)))
        scores["SimRank"].append((u,v, sim[u][v]))

        h_uv = hitting_time(i,j)
        h_vu = hitting_time(j,i)

        scores["Hitting Time"].append((u,v, -h_uv))
        scores["Commute Time"].append((u,v, -(h_uv + h_vu)))

    return scores

In [5]:
# Define ground truth
E_new = set(zip(test_df["src"], test_df["tgt"]))
E_new_star = set([
    (u,v) for (u,v) in E_new
    if u in Core and v in Core
])
n = len(E_new_star)
print(n)

def evaluate(pred_list, E_new_star, n):
    # sort descending
    pred_list = sorted(pred_list, key=lambda x: x[2], reverse=True)
    # keep only Core × Core
    pred_list = [(u,v,s) for u,v,s in pred_list if u in Core and v in Core]
    top_n = pred_list[:n]
    pred_edges = set((u,v) for u,v,_ in top_n)
    # undirected match
    hits = sum((u,v) in E_new_star or (v,u) in E_new_star for u,v in pred_edges)
    return hits

2780


In [6]:
scores = compute_scores()

results = {}
for name, pred in scores.items():
    results[name] = evaluate(pred, E_new_star, n)

for k,v in results.items():
    print(k, v, v/n)

Common Neighbours 24 0.008633093525179856
Jaccard’s coefficient 3 0.001079136690647482
Adamic 23 0.008273381294964029
Preferential Attachment 51 0.018345323741007193
Graph Distance 99 0.03561151079136691
Katz 29 0.010431654676258994
Rooted PageRank 13 0.004676258992805755
SimRank 0 0.0
Hitting Time 15 0.00539568345323741
Commute Time 59 0.021223021582733814
